# Notebook 2: Prepare JUMP-CP standardized dataset

Per `chemical_surrogate_study/PLAN.md`. Toxicity is computed **restricted to active compounds +
negative controls only** (~12,220, not the full ~114,239 population) - the same restriction applied
to BBBC036v1 in notebook 1, justified there by inactive-and-toxic being 3/10,680 = 0.03%. This is
what keeps this notebook's remote data fetches tractable at JUMP's scale.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from huggingface_hub import hf_hub_download
from sklearn.preprocessing import StandardScaler
from copairs import map as copairs_map
from copairs.matching import assign_reference_index

# Assumes the notebook runs with its own directory (chemical_surrogate_study/notebooks/) as the
# working directory, which is Jupyter's default when opening a notebook.
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
STUDY_DIR = PROJECT_ROOT / "chemical_surrogate_study"
DATA_DIR = STUDY_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CELLCLIP_SPLIT_REPO_ID = "suinleelab/CellCLIP"

RANDOM_SEED = 42
SAMPLES_NEGCON = 190
NULL_SIZE = 10000
P_THRESHOLD = 0.05
ACTIVITY_P_THRESHOLD = 0.05
INACTIVE_PADDING_RATIO = 1.0  # 1:1 with active count, per PLAN.md decision

JUMP_RR_RECORD_ID = "20496083"
JUMP_RR_COMPOUND_URL = f"https://zenodo.org/api/records/{JUMP_RR_RECORD_ID}/files/compound.parquet/content"
JUMP_METADATA_COMPOUND_URL = "https://github.com/jump-cellpainting/datasets/raw/main/metadata/compound.csv.gz"
JUMP_ALL_HARMONY_PROFILE_URL = (
    "https://cellpainting-gallery.s3.amazonaws.com/cpg0016-jump-assembled/"
    "source_all/workspace/profiles_assembled/ALL/v1.0a/profiles_var_mad_int_featselect_harmony.parquet"
)
JUMP_INTERPRETABLE_PROFILE_URL = (
    "https://cellpainting-gallery.s3.amazonaws.com/cpg0016-jump-assembled/"
    "source_all/workspace/profiles_assembled/COMPOUND/v1.0/profiles_var_mad_int.parquet"
)
DMSO_JCP2022_ID = "JCP2022_033924"  # confirmed via compound.csv.gz SMILES C[S+](C)[O-] = DMSO
CELLCOUNT_COL = "Nuclei_Number_Object_Number"

print(f"Study dir: {STUDY_DIR}")

## Shared helper functions (same as notebook 1)

In [2]:
def centerscale_on_controls(embeddings, metadata, pert_col, control_key, batch_col=None):
    embeddings = embeddings.copy()
    if batch_col is not None:
        for batch in metadata[batch_col].unique():
            batch_ind = (metadata[batch_col] == batch).to_numpy()
            batch_control_ind = batch_ind & (metadata[pert_col] == control_key).to_numpy()
            embeddings[batch_ind] = StandardScaler().fit(embeddings[batch_control_ind]).transform(embeddings[batch_ind])
        return embeddings
    control_ind = (metadata[pert_col] == control_key).to_numpy()
    return StandardScaler().fit(embeddings[control_ind]).transform(embeddings)


def sample_controls_per_batch(controls_df, batch_col, samples_negcon, random_state):
    return (controls_df.sample(frac=1, random_state=random_state)
            .groupby(batch_col, group_keys=False).head(samples_negcon))


def copairs_phenotypic_activity(features_and_metadata, feature_cols, control_mask, control_query,
                                 control_perturbation, samples_negcon, null_size, random_state,
                                 perturbation_col, batch_col, threshold=P_THRESHOLD, cache_dir=None,
                                 distance="cosine"):
    control_mask = pd.Series(control_mask, index=features_and_metadata.index).fillna(False).astype(bool)
    controls_df = features_and_metadata.loc[control_mask].copy()
    profiles_df = features_and_metadata.loc[~control_mask].copy()
    sampled_controls = sample_controls_per_batch(controls_df, batch_col, samples_negcon, random_state)
    subset = pd.concat([profiles_df, sampled_controls], axis=0).reset_index(drop=True)

    reference_col = "reference_index"
    df_activity = assign_reference_index(subset, control_query, reference_col=reference_col, default_value=-1)
    feature_cols = list(feature_cols)
    features = df_activity[feature_cols].to_numpy(dtype=np.float32, copy=False)
    metadata_cols = df_activity.columns.difference(feature_cols)

    sameby = [perturbation_col, reference_col]
    activity_ap = copairs_map.average_precision(
        df_activity[metadata_cols], features, pos_sameby=sameby, pos_diffby=[],
        neg_sameby=[batch_col], neg_diffby=sameby, distance=distance,
    )
    activity_ap = activity_ap.loc[:, ~activity_ap.columns.duplicated()].copy()
    activity_ap = activity_ap.query(f"{perturbation_col} != @control_perturbation").copy()

    activity_map = copairs_map.mean_average_precision(
        activity_ap, sameby, null_size=null_size, threshold=threshold, seed=random_state,
        cache_dir=Path(cache_dir) if cache_dir is not None else None,
    )
    activity_map = activity_map.query(f"{perturbation_col} != @control_perturbation").copy()
    return activity_map


def load_bbbc_official():
    def load_split(name):
        path = hf_hub_download(CELLCLIP_SPLIT_REPO_ID, f"datasplit1-{name}.csv")
        df = pd.read_csv(path, usecols=["BROAD_ID", "SMILES", "INCHIKEY"])
        df["split"] = name
        return df
    official = pd.concat([load_split(s) for s in ["train", "val", "test"]], axis=0, ignore_index=True)
    return official.drop_duplicates("BROAD_ID").reset_index(drop=True)

## Step 1 — JUMP-RR activity p-values (live fetch from Zenodo)

This *is* "the provided p-values" - no `copairs` rerun for activity (would be too expensive at
JUMP's full scale).

In [3]:
JUMP_RR_CACHE = DATA_DIR / "_jump_rr_raw.parquet"
if JUMP_RR_CACHE.exists():
    jump_rr = pd.read_parquet(JUMP_RR_CACHE)
    print(f"Loaded cached JUMP-RR pull: {len(jump_rr):,} rows")
else:
    t0 = time.time()
    jump_rr = pd.read_parquet(JUMP_RR_COMPOUND_URL, columns=["Perturbation", "JCP2022", "Corrected p-value"])
    jump_rr.to_parquet(JUMP_RR_CACHE)
    print(f"JUMP-RR fetched in {time.time()-t0:.1f}s, {len(jump_rr):,} rows")

jump_rr = jump_rr.drop_duplicates("JCP2022").dropna(subset=["Corrected p-value"]).reset_index(drop=True)
jump_rr["Metadata_JCP2022"] = jump_rr["JCP2022"].astype(str)
jump_rr["inchikey14"] = jump_rr["Perturbation"].astype(str).str.slice(0, 14)
jump_rr["is_active"] = jump_rr["Corrected p-value"] <= ACTIVITY_P_THRESHOLD
jump_rr = jump_rr.rename(columns={"Corrected p-value": "activity_p"})
print(f"Unique compounds: {len(jump_rr):,}, active: {jump_rr['is_active'].sum():,} ({jump_rr['is_active'].mean():.1%})")

JUMP-RR fetched in 36.4s, 2,315,900 rows
Unique compounds: 115,779, active: 12,449 (10.8%)


## Step 2 — SMILES

In [4]:
compound_meta = pd.read_csv(JUMP_METADATA_COMPOUND_URL, compression="gzip")
compound_meta = compound_meta.rename(columns={"Metadata_SMILES": "SMILES"})
jump_rr = jump_rr.merge(compound_meta[["Metadata_JCP2022", "SMILES"]], on="Metadata_JCP2022", how="inner")
jump_rr = jump_rr.dropna(subset=["SMILES"]).reset_index(drop=True)
print(f"With SMILES: {len(jump_rr):,}")

With SMILES: 115,779


## Step 3 — CLOOME/CellCLIP dedup

Drop compounds whose 14-char InChIKey matches BBBC036v1's official population (self-contained -
recomputed directly, not the external pickle the old script used). Conservative: dedups against
the full official train+val+test population, not just CLOOME/CellCLIP's narrower training split.

In [5]:
bbbc_official = load_bbbc_official()
bbbc_official["inchikey14"] = bbbc_official["INCHIKEY"].astype(str).str.slice(0, 14)
bbbc_inchikeys = set(bbbc_official["inchikey14"])

n_before = len(jump_rr)
jump_rr = jump_rr[~jump_rr["inchikey14"].isin(bbbc_inchikeys)].reset_index(drop=True)
print(f"Deduped against BBBC036v1 official population: {n_before:,} -> {len(jump_rr):,} ({n_before - len(jump_rr)} removed)")

Deduped against BBBC036v1 official population: 115,779 -> 114,239 (1540 removed)


## Step 4 — toxicity, restricted to active compounds + negative controls

Interpretable product `COMPOUND/v1.0/profiles_var_mad_int.parquet`. Confirmed: no
`Metadata_pert_type` column exists in JUMP's metadata anywhere; DMSO controls are identified by
`Metadata_JCP2022 == "JCP2022_033924"` (verified via SMILES `C[S+](C)[O-]`). This column is
already variance/MAD-normalized by JUMP; we still apply our own per-plate DMSO-centering on top for
exact methodological consistency with BBBC.

In [6]:
active_jcp_ids = jump_rr.loc[jump_rr["is_active"], "Metadata_JCP2022"].tolist()
print(f"Fetching cell count for {len(active_jcp_ids):,} active compounds...")

CELLCOUNT_CACHE = DATA_DIR / "_jump_cellcount_active_wells.parquet"
if CELLCOUNT_CACHE.exists():
    active_cc_wells = pl.read_parquet(CELLCOUNT_CACHE).to_pandas()
else:
    t0 = time.time()
    active_ids_df = pl.DataFrame({"Metadata_JCP2022": active_jcp_ids}).lazy()
    cols = ["Metadata_JCP2022", "Metadata_Plate", "Metadata_Well", "Metadata_Source", CELLCOUNT_COL]
    active_cc_wells_pl = (pl.scan_parquet(JUMP_INTERPRETABLE_PROFILE_URL)
                          .select(cols)
                          .join(active_ids_df, on="Metadata_JCP2022", how="semi")
                          .collect())
    active_cc_wells = active_cc_wells_pl.to_pandas()
    active_cc_wells.to_parquet(CELLCOUNT_CACHE)
    print(f"Active-compound cell count fetched in {time.time()-t0:.1f}s, {len(active_cc_wells):,} wells")

relevant_plates = active_cc_wells["Metadata_Plate"].unique().tolist()
print(f"{len(relevant_plates):,} relevant plates")

DMSO_CACHE = DATA_DIR / "_jump_cellcount_dmso_wells.parquet"
if DMSO_CACHE.exists():
    dmso_wells = pl.read_parquet(DMSO_CACHE).to_pandas()
else:
    t0 = time.time()
    plates_df = pl.DataFrame({"Metadata_Plate": relevant_plates}).lazy()
    cols = ["Metadata_JCP2022", "Metadata_Plate", "Metadata_Well", "Metadata_Source", CELLCOUNT_COL]
    dmso_wells_pl = (pl.scan_parquet(JUMP_INTERPRETABLE_PROFILE_URL)
                     .select(cols)
                     .filter(pl.col("Metadata_JCP2022") == DMSO_JCP2022_ID)
                     .join(plates_df, on="Metadata_Plate", how="semi")
                     .collect())
    dmso_wells = dmso_wells_pl.to_pandas()
    dmso_wells.to_parquet(DMSO_CACHE)
    print(f"DMSO wells on relevant plates fetched in {time.time()-t0:.1f}s, {len(dmso_wells):,} wells")

active_cc_wells["is_control"] = False
active_cc_wells["perturbation"] = active_cc_wells["Metadata_JCP2022"]
dmso_wells["is_control"] = True
dmso_wells["perturbation"] = "DMSO"

cc_wells_all = pd.concat([active_cc_wells, dmso_wells], axis=0, ignore_index=True)
cc_wells_all = cc_wells_all.dropna(subset=[CELLCOUNT_COL]).reset_index(drop=True)
print(f"Combined cell-count wells (active + DMSO): {len(cc_wells_all):,}")

Fetching cell count for 12,170 active compounds...


Active-compound cell count fetched in 9.7s, 84,537 wells
1,668 relevant plates


DMSO wells on relevant plates fetched in 9.4s, 76,977 wells
Combined cell-count wells (active + DMSO): 161,514


In [7]:
cellcount_std = centerscale_on_controls(cc_wells_all[[CELLCOUNT_COL]].to_numpy(dtype=np.float64), cc_wells_all,
                                         "is_control", True, batch_col="Metadata_Plate")
cc_wells_std = pd.concat([
    cc_wells_all[["Metadata_JCP2022", "Metadata_Plate", "Metadata_Well", "is_control", "perturbation"]].reset_index(drop=True),
    pd.DataFrame(cellcount_std.astype(np.float32), columns=[CELLCOUNT_COL]),
], axis=1)

t0 = time.time()
toxic_map = copairs_phenotypic_activity(
    cc_wells_std, feature_cols=[CELLCOUNT_COL], control_mask=cc_wells_std["is_control"],
    control_query="is_control == True", control_perturbation="DMSO",
    samples_negcon=SAMPLES_NEGCON, null_size=NULL_SIZE, random_state=RANDOM_SEED,
    perturbation_col="perturbation", batch_col="Metadata_Plate",
    cache_dir=DATA_DIR / "_jump_toxicity_null_cache",
    distance="euclidean",
)
print(f"Toxic-label copairs test done in {time.time()-t0:.1f}s")
toxic_labels = toxic_map.drop_duplicates("perturbation")[
    ["perturbation", "mean_average_precision", "corrected_p_value", "below_corrected_p"]
].rename(columns={"perturbation": "Metadata_JCP2022", "mean_average_precision": "toxicity_map",
                   "corrected_p_value": "toxicity_p", "below_corrected_p": "is_toxic"})
print(f"Toxic (among actives): {toxic_labels['is_toxic'].sum():,} / {len(toxic_labels):,} ({toxic_labels['is_toxic'].mean():.1%})")

  0%|          | 0/88 [00:00<?, ?it/s]

  0%|          | 0/285 [00:00<?, ?it/s]

  0%|          | 0/439 [00:00<?, ?it/s]

  0%|          | 1/439 [00:01<07:25,  1.02s/it]

 10%|█         | 44/439 [00:01<00:07, 50.01it/s]

...

 81%|████████  | 354/439 [00:02<00:00, 190.42it/s]

 90%|█████████ | 397/439 [00:03<00:00, 129.54it/s]

  0%|          | 0/12170 [00:00<?, ?it/s]

 79%|███████▉  | 9655/12170 [00:00<00:00, 96455.35it/s]

Toxic-label copairs test done in 9.2s
Toxic (among actives): 5,758 / 12,170 (47.3%)


## Step 5 — define the final population: all active ∪ inactive padding

In [8]:
jump_rr = jump_rr.merge(toxic_labels, on="Metadata_JCP2022", how="left")
jump_rr["is_toxic"] = jump_rr["is_toxic"].fillna(False)
jump_rr["is_active_and_toxic"] = jump_rr["is_active"] & jump_rr["is_toxic"]
jump_rr["is_active_not_toxic"] = jump_rr["is_active"] & ~jump_rr["is_toxic"]

active_pop = jump_rr[jump_rr["is_active"]].copy()
inactive_pop_all = jump_rr[~jump_rr["is_active"]].copy()
print(f"Active: {len(active_pop):,}, Inactive pool: {len(inactive_pop_all):,}")

# Lightweight Metadata_Source lookup for the inactive pool (2 narrow columns, no feature data)
SOURCE_LOOKUP_CACHE = DATA_DIR / "_jump_source_lookup.parquet"
if SOURCE_LOOKUP_CACHE.exists():
    source_lookup = pd.read_parquet(SOURCE_LOOKUP_CACHE)
else:
    t0 = time.time()
    source_lookup_pl = (pl.scan_parquet(JUMP_ALL_HARMONY_PROFILE_URL)
                        .select(["Metadata_JCP2022", "Metadata_Source"])
                        .unique(subset=["Metadata_JCP2022"], keep="first")
                        .collect())
    source_lookup = source_lookup_pl.to_pandas()
    source_lookup.to_parquet(SOURCE_LOOKUP_CACHE)
    print(f"Source lookup fetched in {time.time()-t0:.1f}s, {len(source_lookup):,} compounds")

active_pop = active_pop.merge(source_lookup, on="Metadata_JCP2022", how="left")
inactive_pop_all = inactive_pop_all.merge(source_lookup, on="Metadata_JCP2022", how="left")

n_inactive_target = int(len(active_pop) * INACTIVE_PADDING_RATIO)
inactive_pop_all = inactive_pop_all.dropna(subset=["Metadata_Source"])
inactive_pop = (inactive_pop_all.groupby("Metadata_Source", group_keys=False)
                .apply(lambda g: g.sample(
                    n=min(len(g), max(1, round(n_inactive_target * len(g) / len(inactive_pop_all)))),
                    random_state=RANDOM_SEED)))
print(f"Inactive padding sampled: {len(inactive_pop):,} (target was {n_inactive_target:,})")

final_population = pd.concat([active_pop, inactive_pop], axis=0, ignore_index=True)
print(f"Final population: {len(final_population):,} ({final_population['is_active'].mean():.1%} active)")
print(final_population[["is_active", "is_active_and_toxic", "is_active_not_toxic"]].mean())

/var/folders/ch/2fy5zt9x53vfq0jsm_n8ml0w0000gn/T/ipykernel_63310/3290202663.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  jump_rr["is_toxic"] = jump_rr["is_toxic"].fillna(False)


Active: 12,170, Inactive pool: 102,069


Source lookup fetched in 8.1s, 138,903 compounds
Inactive padding sampled: 12,171 (target was 12,170)
Final population: 24,341 (50.0% active)
is_active              0.499979
is_active_and_toxic    0.236556
is_active_not_toxic    0.263424
dtype: float64


/var/folders/ch/2fy5zt9x53vfq0jsm_n8ml0w0000gn/T/ipykernel_63310/3290202663.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(


## Step 6 — fetch morphology, restricted to the final population

In [9]:
MORPH_CACHE = DATA_DIR / "_jump_morphology_final_pop.parquet"
final_ids = final_population["Metadata_JCP2022"].tolist()

if MORPH_CACHE.exists():
    morph_df = pl.read_parquet(MORPH_CACHE).to_pandas()
else:
    t0 = time.time()
    schema_cols = pl.scan_parquet(JUMP_ALL_HARMONY_PROFILE_URL).collect_schema().names()
    jump_feature_cols = [c for c in schema_cols if not c.startswith("Metadata_")]
    ids_df = pl.DataFrame({"Metadata_JCP2022": final_ids}).lazy()

    morph_lazy = (pl.scan_parquet(JUMP_ALL_HARMONY_PROFILE_URL)
                 .join(ids_df, on="Metadata_JCP2022", how="semi")
                 .group_by("Metadata_JCP2022")
                 .agg([pl.col(c).median().alias(c) for c in jump_feature_cols]))
    morph_df = morph_lazy.collect().to_pandas()
    morph_df.to_parquet(MORPH_CACHE)
    print(f"Morphology fetched in {time.time()-t0:.1f}s, {len(morph_df):,} compounds x {len(jump_feature_cols)} features")

print(f"Morphology coverage: {morph_df['Metadata_JCP2022'].isin(final_ids).sum():,} / {len(final_ids):,}")

Morphology fetched in 347.8s, 24,341 compounds x 722 features
Morphology coverage: 24,341 / 24,341


## Step 7 — assemble and write

In [10]:
morph_feature_cols = [c for c in morph_df.columns if c != "Metadata_JCP2022"]
morph_indexed = morph_df.set_index("Metadata_JCP2022")

final_population = final_population[final_population["Metadata_JCP2022"].isin(morph_indexed.index)].reset_index(drop=True)
morph_matrix = morph_indexed.reindex(final_population["Metadata_JCP2022"]).to_numpy(dtype=np.float32)
morph_matrix = np.nan_to_num(morph_matrix, nan=0.0, posinf=0.0, neginf=0.0)
morph_cols = [f"morph_{i}" for i in range(morph_matrix.shape[1])]
morph_out_df = pd.DataFrame(morph_matrix, columns=morph_cols, index=final_population.index)

standardized = pd.concat([
    final_population.rename(columns={"Metadata_JCP2022": "compound_id", "Metadata_Source": "batch_id"})[
        ["compound_id", "SMILES", "is_active", "is_active_and_toxic", "is_active_not_toxic",
         "activity_p", "toxicity_map", "toxicity_p", "batch_id"]
    ].assign(activity_map=np.nan, cloome_split=None),
    morph_out_df,
], axis=1)
# reorder to match the standardized schema exactly (activity_map is unavailable from JUMP-RR - only p-value is published)
standardized = standardized[["compound_id", "SMILES", "is_active", "is_active_and_toxic", "is_active_not_toxic",
                              "activity_map", "activity_p", "toxicity_map", "toxicity_p", "batch_id", "cloome_split"] + morph_cols]

print(f"Final standardized population: {len(standardized):,} compounds, {len(morph_cols)} morphology features")
print(standardized[["is_active", "is_active_and_toxic", "is_active_not_toxic"]].mean())
print(f"Sources represented: {final_population['Metadata_Source'].nunique()}")

OUT_PATH = DATA_DIR / "jumpcp_standardized.parquet"
standardized.to_parquet(OUT_PATH)
print(f"\nSaved -> {OUT_PATH}")

Final standardized population: 24,341 compounds, 722 morphology features
is_active              0.499979
is_active_and_toxic    0.236556
is_active_not_toxic    0.263424
dtype: float64
Sources represented: 8



Saved -> /Users/telio/chemical-surrogate-limits/chemical_surrogate_study/data/jumpcp_standardized.parquet
